# I - Préliminaires - Fonctions utiles

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import itertools
import numpy as np

def plot_confusion_matrix(cm, classes, normalize=False,
                          title='Matrice de confusion', cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(8,8))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('Vraie classe')
    plt.xlabel('Classe prédite')
    plt.show()

# II - Entraînement d'un CNN pour la classification sur CIFAR10

### II.1. Chargement et Dimensionnement de la base CIFAR10

In [ ]:
from keras import backend as K
print(K.backend())

In [ ]:
from keras.datasets import cifar10
(x_train_full, y_train_full), (x_test_full, y_test_full) = cifar10.load_data()
print("Dimension de la base d'apprentissage CIFAR10 :", x_train_full.shape)
print("Dimension des vecteurs d'étiquette de classe :", y_train_full.shape)
print("Dimension de la base de test CIFAR10 :", x_test_full.shape)

In [ ]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Réduction de la taille du dataset et standardisation des données

In [ ]:
n_training_samples = 5000
n_other_samples = 2000

def standardize(img_data):
    """Normalise chaque image : soustrait la moyenne et divise par l'écart-type."""
    img_data_mean = np.mean(img_data, axis=(1,2), keepdims=True)
    img_data_std  = np.std(img_data,  axis=(1,2), keepdims=True)
    return (img_data - img_data_mean) / img_data_std

train_ids = np.random.choice(len(x_train_full), size=n_training_samples, replace=False)
other_ids = np.random.choice(len(x_test_full),  size=n_other_samples,    replace=False)

n_valid  = n_other_samples // 2
val_ids  = other_ids[:n_valid]
test_ids = other_ids[n_valid:]

x_train_initial = x_train_full[train_ids]
x_val_initial   = x_test_full[val_ids]
x_test_initial  = x_test_full[test_ids]

x_train = standardize(x_train_initial.astype('float32'))
x_val   = standardize(x_val_initial.astype('float32'))
x_test  = standardize(x_test_initial.astype('float32'))

y_train = y_train_full[train_ids]
y_val   = y_test_full[val_ids]
y_test  = y_test_full[test_ids]

print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

---
### Réponses – Section 2 : Structuration des données

**Q1 – Pourquoi diviser en trois sous-ensembles {train, validation, test} ?**

- **Train** : sert à ajuster les poids du réseau (gradient descent).
- **Validation** : sert à mesurer la performance pendant l'entraînement et à choisir les hyperparamètres (learning rate, taille du réseau…) sans contaminer le test.
- **Test** : évalue la performance finale sur des données jamais vues, pour estimer la capacité de généralisation réelle. Si on utilisait le test pour choisir les hyperparamètres, on sur-estimerait les performances.

**Q2 – Que fait `standardize` ? Quel est son intérêt ?**

`standardize` soustrait la moyenne pixelaire de chaque image et divise par son écart-type, ce qui ramène chaque image à une distribution de moyenne 0 et d'écart-type 1. Cela améliore le conditionnement de l'optimisation : les gradients ont des magnitudes similaires pour tous les canaux, ce qui accélère la convergence et stabilise l'apprentissage.

In [ ]:
# Affichage de quelques images d'entraînement
n_display = 12
random_ids = np.random.choice(len(x_train), n_display, replace=False)
f, axarr = plt.subplots(1, n_display, figsize=(16,2))
for k in range(n_display):
    axarr[k].imshow(x_train_initial[random_ids[k]])
    axarr[k].title.set_text(classes[y_train[random_ids[k]][0]])
    axarr[k].axis('off')
plt.show()

In [ ]:
from keras.utils import to_categorical
y_train = to_categorical(y_train)
y_val   = to_categorical(y_val)
y_test  = to_categorical(y_test)
print("y_train:", y_train.shape, "y_val:", y_val.shape, "y_test:", y_test.shape)

### II.2. Définition de l'architecture du CNN

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout, Activation, BatchNormalization
from keras.regularizers import l2
from keras import Input

# ── Modèle amélioré (Section 6) : 3 blocs conv + dropout + batchnorm ──────────
model = Sequential([
    Input(shape=(32, 32, 3)),

    # Bloc 1
    Conv2D(32, (3,3), activation='relu', padding='same', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    MaxPool2D(pool_size=(2,2)),   # 32x32 -> 16x16
    Dropout(0.25),

    # Bloc 2
    Conv2D(64, (3,3), activation='relu', padding='same', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    MaxPool2D(pool_size=(2,2)),   # 16x16 -> 8x8
    Dropout(0.25),

    # Classificateur
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax'),
])

weights_init = model.get_weights()

---
### Réponses – Section 3 : Architecture du réseau

**Rôle de chaque type de couche :**
- **Conv2D** : applique des filtres apprenables pour détecter des motifs locaux (bords, textures…). `filters` = nombre de filtres, `kernel_size` = taille du filtre, `activation` = non-linéarité, `padding='same'` conserve la dimension spatiale.
- **BatchNormalization** : normalise les activations du batch courant, stabilise et accélère l'apprentissage.
- **MaxPool2D** : sous-échantillonne la feature map en gardant la valeur maximale dans chaque fenêtre → réduit la taille spatiale d'un facteur 2.
- **Dropout** : désactive aléatoirement une fraction des neurones à chaque batch (régularisation contre le sur-apprentissage).
- **Flatten** : aplatit le tenseur 3D en vecteur 1D pour passer aux couches denses.
- **Dense** : couche fully-connected ; la dernière avec `softmax` produit une distribution de probabilité sur les 10 classes.

**Tailles des feature maps (modèle de base simple avec 1 conv 8 filtres) :**

| Couche | Sortie (H × W × C) |
|--------|--------------------|
| Entrée | 32 × 32 × 3 |
| Conv2D(8, 3×3, same) | 32 × 32 × 8 |
| MaxPool2D(2×2) | 16 × 16 × 8 |
| Flatten | 2048 |
| Dense(10, softmax) | 10 |

**Codage de la sortie :** `to_categorical` encode la classe sous forme de vecteur one-hot de taille 10 (ex. classe 3 → [0,0,0,1,0,0,0,0,0,0]).

**Q1 – Trainable params : comment sont-ils calculés ?**

Pour une Conv2D avec `F` filtres de taille `k×k` sur `C` canaux : `F × (k×k×C + 1)` paramètres (+1 pour le biais).
Pour une Dense avec `n_in` entrées et `n_out` sorties : `n_in × n_out + n_out` paramètres.
Les couches MaxPool, Flatten n'ont pas de paramètres apprenables.

### II.3. Définition de la fonction de coût et choix de l'optimiseur

In [ ]:
from keras.optimizers import Adam, SGD

# Adam offre en général une meilleure convergence que SGD simple
opt = Adam(learning_rate=0.001)
# opt = SGD(learning_rate=0.01, momentum=0.9)  # SGD avec momentum

model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['acc'])
print(model.summary())

### II.4. Entraînement du CNN

In [ ]:
from keras.callbacks import Callback, ModelCheckpoint
import time

class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.times = []
    def on_epoch_begin(self, batch, logs={}):
        self.epoch_time_start = time.time()
    def on_epoch_end(self, batch, logs={}):
        self.times.append(time.time() - self.epoch_time_start)

time_callback = TimeHistory()
filepath   = "my_model.weights.h5"
checkpoint = ModelCheckpoint(filepath, monitor='val_acc', save_best_only=True, verbose=1)
callbacks  = [time_callback, checkpoint]

In [ ]:
history = model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=30,
    verbose=1,
    validation_data=(x_val, y_val),
    callbacks=callbacks
)

---
### Réponses – Section 4 : Apprentissage

**Q1 – Époque, étape, lot :**
- **Lot (batch)** : sous-ensemble d'exemples utilisé pour calculer un gradient et faire une mise à jour des poids.
- **Étape (step)** : une passe forward + backward sur un batch. Nombre d'étapes par époque = `n_train / batch_size`.
- **Époque (epoch)** : une passe complète sur l'ensemble d'entraînement (toutes les étapes).

**Q2 – Effet du batch size :**
- **Petit batch** : étape rapide, gradient bruité (meilleure généralisation mais convergence moins stable).
- **Grand batch** : étape plus longue, gradient lissé (convergence stable mais risque de sur-apprentissage et besoin de plus de mémoire). Le nombre d'étapes par époque diminue, donc l'époque est plus rapide en nombre de steps mais chaque step coûte plus cher.

**Q3 – Comparaison SGD / SGD+Momentum / Adam :**
- **SGD** : simple, lent, sensible au learning rate.
- **SGD + Momentum** : accumule les gradients passés → convergence plus rapide et moins d'oscillations.
- **Adam** : adapte le learning rate pour chaque paramètre individuellement. En général le plus rapide à converger et le moins sensible au réglage du learning rate. C'est le choix par défaut recommandé.

In [ ]:
# Statistiques de temps
times = time_callback.times
print("Temps moyen par époque : {:.2f}s".format(np.mean(times)))
print("Écart-type             : {:.2f}s".format(np.std(times)))

In [ ]:
# Tracé des courbes d'apprentissage
history_dict    = history.history
loss_values     = history_dict['loss']
val_loss_values = history_dict['val_loss']
acc_values      = history_dict['acc']
val_acc_values  = history_dict['val_acc']
epochs_range    = range(1, len(acc_values) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, loss_values,     'b', label='Train loss')
ax1.plot(epochs_range, val_loss_values, 'g', label='Val loss')
ax1.set_title('Fonctions de coût')
ax1.set_xlabel('Époques'); ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(epochs_range, acc_values,     'b', label='Train acc')
ax2.plot(epochs_range, val_acc_values, 'g', label='Val acc')
ax2.set_title('Précision')
ax2.set_xlabel('Époques'); ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

---
### Réponses – Section 5 : Hyperparamètres

**Hyperparamètres du modèle et leurs effets :**

| Hyperparamètre | Nature | Effet |
|---|---|---|
| `learning_rate` | Scalaire (float) | Taille des pas de gradient. Trop grand → divergence. Trop petit → convergence lente. |
| `batch_size` | Entier | Compromis vitesse/stabilité du gradient. |
| `epochs` | Entier | Nombre de passages sur les données. Trop peu → sous-apprentissage. Trop → sur-apprentissage. |
| `filters` (Conv2D) | Entier | Nombre de motifs détectés. Plus de filtres = plus de capacité, plus de calculs. |
| `kernel_size` | Tuple | Taille du champ récepteur local. 3×3 est le standard. |
| `dropout rate` | Float [0,1] | Force de la régularisation. 0.25–0.5 typique. |
| `l2` | Float | Pénalise les grands poids (régularisation L2). |
| `n_training_samples` | Entier | Taille de la base d'entraînement. Plus grand = meilleure généralisation. |
| Optimiseur | Choix | SGD, Adam… influence la vitesse de convergence. |

### II.5. Reprise depuis checkpoint (optionnel)

In [ ]:
from keras.models import load_model
import pathlib

file = pathlib.Path(filepath)
if file.exists():
    model.load_weights(filepath)
    model.compile(optimizer=Adam(learning_rate=0.0005),
                  loss='categorical_crossentropy', metrics=['acc'])
    history_2 = model.fit(x_train, y_train, batch_size=64, epochs=10,
                          verbose=1, validation_data=(x_val, y_val),
                          callbacks=[time_callback, checkpoint])
else:
    print("Pas de checkpoint trouvé, on continue avec le modèle courant.")

# III - Test et Évaluation du modèle

In [ ]:
# Prédictions sur quelques images de test
n_display  = 12
random_ids = np.random.choice(len(x_test), n_display, replace=False)
pred       = np.argmax(model.predict(x_test[random_ids]), axis=1)

f, axarr = plt.subplots(1, n_display, figsize=(16, 2))
for k in range(n_display):
    axarr[k].imshow(x_test_initial[random_ids[k]])
    axarr[k].title.set_text(classes[pred[k]])
    axarr[k].axis('off')
plt.show()

In [ ]:
print("Précision train : {:.2f} %".format(100 * history_dict['acc'][-1]))
print("Précision val   : {:.2f} %".format(100 * history_dict['val_acc'][-1]))

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("Précision test  : {:.2f} %".format(100 * test_acc))

In [ ]:
# Matrice de confusion
n_classes      = len(classes)
confusion_matrix = np.zeros((n_classes, n_classes), dtype=np.int64)
pred_all       = np.argmax(model.predict(x_test), axis=1)

for i in range(len(y_test)):
    confusion_matrix[np.argmax(y_test[i]), pred_all[i]] += 1

print("{:<12} {:^12}".format("Classe", "Précision (%)"))
for i in range(n_classes):
    total   = confusion_matrix[i, :].sum()
    correct = confusion_matrix[i, i]
    print("{:<12} {:^12.1f}".format(classes[i], 100 * correct / total))

### III.2. Matrices de Confusion

In [ ]:
plot_confusion_matrix(confusion_matrix, classes, normalize=True,
                      title='Matrice de confusion normalisée')
plot_confusion_matrix(confusion_matrix, classes,
                      title='Matrice de confusion non normalisée')

---
### Réponses – Section 6 : Approfondissement du modèle

**Q2 – Récapitulatif du modèle final :**
- 2 blocs de 2 Conv2D (32 puis 64 filtres, kernel 3×3, padding same, ReLU)
- BatchNormalization après chaque conv
- MaxPool2D(2×2) + Dropout(0.25) entre les blocs
- Dense(256, ReLU) + Dropout(0.5) + Dense(10, softmax)
- Optimiseur : Adam(lr=0.001), loss : categorical_crossentropy

**Q3 – Critique du modèle :**
- **Avantages** : plus performant qu'un modèle à 1 couche (meilleure extraction de features hiérarchiques), rapide à entraîner sur CPU grâce à la petite taille.
- **Limitations** : seulement 5000 images d'entraînement → performance limitée (~65-70%). Un modèle entraîné sur 50k images avec data augmentation atteindrait 85-90%.

---
### Réponses – Section 7 : Sur-apprentissage

**Q1 – Comment observer l'overfitting sur les courbes ?**

Le sur-apprentissage est visible quand la **loss de validation augmente** (ou stagne) alors que la **loss d'entraînement continue de diminuer**. De même, l'accuracy train monte mais l'accuracy val plafonne ou baisse. Typiquement visible après quelques époques avec un petit dataset.

**Q2 – Mécanismes contre le sur-apprentissage :**
- **Dropout** : désactive aléatoirement des neurones → le réseau ne peut pas mémoriser les exemples.
- **Régularisation L2** (`kernel_regularizer=l2(...)`) : pénalise les grands poids dans la loss.
- **BatchNormalization** : stabilise les activations, a un effet régularisant.
- **Data augmentation** : augmente artificiellement la diversité des données (flips, crops…).
- **Early stopping** : arrêter l'entraînement quand la val_loss remonte.

# IV - Visualisation des cartes d'activation

In [ ]:
from keras.models import Model

# Modèle tronqué : sortie de la 1ère couche conv (index 1 = après Input)
reduced_model = Model(inputs=model.inputs, outputs=model.layers[1].output)
reduced_model.summary()

In [ ]:
feature_maps = reduced_model.predict(x_test)

def get_mask(k):
    fm_positive = np.maximum(feature_maps[k], 0)
    mask = np.sum(fm_positive, axis=2)
    mask = mask / (np.max(mask) + 1e-8)
    return mask

random_ids = np.random.choice(len(x_test), n_display, replace=False)

f, rd_img = plt.subplots(1, n_display, figsize=(16,2))
for k in range(n_display):
    rd_img[k].imshow(x_test_initial[random_ids[k]])
    rd_img[k].axis('off')
plt.suptitle("Images originales")
plt.show()

f, rd_maps = plt.subplots(1, n_display, figsize=(16,2))
for k in range(n_display):
    mask = get_mask(random_ids[k])
    rd_maps[k].imshow(mask, cmap='hot')
    rd_maps[k].axis('off')
plt.suptitle("Cartes d'activation (couche 1)")
plt.show()

---
### Réponses – Section 8 : Cartes d'activation

**Q1 – Interprétation des cartes d'activation de la 1ère couche :**

Les cartes d'activation de la première couche convolutive mettent en évidence les **zones de l'image qui activent fortement les filtres** appris. Les filtres de la 1ère couche détectent généralement des **motifs de bas niveau** : contours, gradients de couleur, orientations. Les zones brillantes sur la carte correspondent aux régions qui contiennent ces motifs. En début d'entraînement, les activations sont quasi-aléatoires ; après convergence, elles se concentrent sur les structures visuelles importantes.

**Q2 – Cartes des couches plus profondes :**

Les couches profondes activent sur des **motifs de plus haut niveau** (formes, parties d'objets). La carte est moins "texturée" et plus sélective : peu de zones très activées, correspondant aux parties discriminantes de l'objet (ex : contour de l'aile d'un avion, silhouette d'un animal).

**Q3 – Évolution des cartes au cours de l'entraînement :**

- **Au début** : activations diffuses, aléatoires, sans structure claire.
- **En milieu d'entraînement** : les filtres commencent à se spécialiser, certaines zones ressortent.
- **En fin d'entraînement** : les cartes sont plus nettes et sélectives, les filtres ont appris des détecteurs de motifs pertinents pour la classification.